# 00 Sanity Check — 10-Prompt Baseline (Condition A)

This notebook validates the Condition A pipeline on 10 prompts before the full 50-prompt run.

**Purpose:** Gate G2 — Baseline established and verified

In [ ]:
import sys, os
sys.path.insert(0, '../src')
import json, numpy as np
from conditions import generate_sl
from evaluate import evaluate_result
from utils import load_config, bootstrap_ci

cfg = load_config('../configs/experiment.yaml')
model_cfg = cfg['model']
print('Config loaded:', cfg['experiment']['name'])

In [ ]:
# Load first 10 prompts from HypoBench
with open('../data/HypoBench/hypobench_50.json') as f:
    all_prompts = json.load(f)
prompts_10 = all_prompts[:10]
print(f'Loaded {len(prompts_10)} prompts for sanity check')
print('Example prompt ID:', prompts_10[0]['id'])

In [ ]:
# Run Condition A on 10 prompts
outputs, scores = [], []
for entry in prompts_10:
    result = generate_sl(prompt=entry['gap'], background=entry['background'], n=3, cfg=model_cfg)
    score  = evaluate_result(result=result, background=entry['background'],
                             gap=entry['gap'], reference=entry['reference_hypothesis'], cfg=model_cfg)
    outputs.append(result)
    scores.append(score)
    print(f"[{entry['id']}] Q={score['quality']:.2f}  tokens={score['tokens']}")

In [ ]:
# Sanity check summary
q_vals = [s['quality'] for s in scores]
g_vals = [s['groundedness'] for s in scores]
t_vals = [s['tokens'] for s in scores]

print('=== Sanity Check Results (n=10) ===')
print(f'All outputs parseable:        {all(len(o["hypotheses"]) == 3 for o in outputs)} (expected True)')
print(f'Groundedness mean:            {np.mean(g_vals):.2f} (expected > 0.5)')
print(f'Quality judge mean:           {np.mean(q_vals):.2f} / 5.0')
print(f'Quality 95% CI:               {bootstrap_ci(q_vals)}')
print(f'Token count mean/prompt:      {np.mean(t_vals):.0f}')
print()
print('BASELINE VERIFIED [OK]  -- Full 50-prompt run approved')